# **Q/A Chatbot**

### Basic working of Google Palm LLM in LangChain

In [10]:
#from langchain.llms import GooglePalm
#api_key = 'your api key here' # get this free api key from https://makersuite.google.com/
#llm = GooglePalm(google_api_key=api_key, temperature=0.1)

from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI 


load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("BASE_URL")

llm = init_chat_model("qwen3.8:latest", model_provider="ollama", temperature=0)
llm = ChatOpenAI(api_key=api_key,
                        base_url=base_url,
                        model="gapgpt-qwen-3.5",
                        temperature=0.7,) 


In [ ]:
poem = llm.invoke("Write a 4 line poem of my love for samosa")
print(poem.content)

content='A golden shell, a spiced heart within,\nEach bite a crunch, a warm, savory sin,\nNo pastry, no pie, can steal my crown—\nMy love for samosa, forever bound.' additional_kwargs={} response_metadata={'model': 'qwen3.8:latest', 'created_at': '2026-09-09T04:22:59.118707Z', 'done': True, 'done_reason': 'stop', 'total_duration': 28345233600, 'load_duration': 17017088700, 'prompt_eval_count': 22, 'prompt_eval_duration': 633075000, 'eval_count': 193, 'eval_duration': 10692686000, 'logprobs': None, 'model_name': 'qwen3.8:latest', 'model_provider': 'ollama'} id='lc_run--01a08467-3270-7ea1-86eb-0c8c2efdee65-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 22, 'output_tokens': 193, 'total_tokens': 215}


In [2]:
essay = llm.invoke("write email requesting refund for electronic item")
print(essay.content)

Subject: Refund Request – Order #[Order Number]

Dear [Store/Company Name] Customer Service Team,

I hope this message finds you well. I am writing to request a refund for an electronic item I purchased from your store.

**Order Details:**
- Order Number: [Order #]
- Date of Purchase: [Date]
- Item: [e.g., Sony WH-1000XM5 Wireless Headphones]
- Amount Paid: $[Amount]
- Payment Method: [Credit Card / PayPal / etc.]

**Reason for Refund:**
[Choose or adapt one of the following:]

- *Defective:* The item arrived with a manufacturing defect. [Specifically, the left earbud produces no sound / the screen flickers intermittently / the device will not hold a charge beyond 10 minutes.]
- *Not as described:* The product does not match the specifications listed on your website. [e.g., The listing stated 256 GB storage, but the device only contains 128 GB.]
- *Damaged in transit:* The item arrived with visible damage. [e.g., The screen is cracked and the casing is dented on the left side.]
- *Chan

In [3]:
#from langchain.chains import RetrievalQA
#from langchain.embeddings import GooglePalmEmbeddings
#from langchain.llms import GooglePalm

### Now let's load data from Codebasics FAQ csv file

In [2]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(file_path='codebasics_faqs.csv', source_column="prompt")

# Store the loaded data in the 'data' variable
data = loader.load()

C:\Users\meisa\AppData\Local\Temp\ipykernel_43952\1741362776.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader
W0909 09:09:53.495000 43952 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0909 09:09:53.657000 43952 Lib\site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


### Hugging Face Embeddings

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize instructor embeddings using the Hugging Face model
instructor_embeddings = HuggingFaceEmbeddings(model_name="hkunlp/instructor-large")

e = instructor_embeddings.embed_query("What is your refund policy?")

In [4]:
print(len(e))
e[:5]

768


[-0.04449571669101715,
 0.007691530976444483,
 -0.009869121015071869,
 0.020831888541579247,
 0.03185895085334778]

As you can see above, embedding for a sentance "What is your refund policy" is a list of size 768. Looking at the numbers in this list, doesn't give any intuitive understanding of what it is but just assume that these numbers are capturing the meaning of "What is your refund policy". If you are curious to know about embeddings, go to youtube and search "codebasics word embeddings" and you will find bunch of videos with simple, intuitive explanations

### Vector store using FAISS

In [5]:
from langchain_community.vectorstores import FAISS

# Create a FAISS instance for vector database from 'data'
vectordb = FAISS.from_documents(documents=data,
                                 embedding=instructor_embeddings)

# Create a retriever for querying the vector database
retriever = vectordb.as_retriever(score_threshold = 0.7)

In [6]:
save=False
file_path = "faiss_index_file"  # This will be a directory

if save == True:
    # ✅ CORRECT: Use FAISS's built-in save method
    vectordb.save_local(file_path)
    print(f"✅ Vector store saved to: {file_path}")
else:
    from langchain_community.vectorstores import FAISS

    loaded_vectorstore = FAISS.load_local(
        file_path,
        embeddings=instructor_embeddings,  # You need the same embeddings instance
        allow_dangerous_deserialization=True  # Required for FAISS
    )
    print(f"✅ Vector store loaded from: {file_path}")

✅ Vector store loaded from: faiss_index_file


In [7]:
rdocs = retriever.invoke("how about job placement support?")

for i, doc in enumerate(rdocs):
    print(f"Result {i+1}:")
    print(f"Content: {doc.page_content[:200]}...")
    print(f"Metadata: {doc.metadata}")
    print("-" * 40)

#rdocs

Result 1:
Content: prompt: Do you provide any job assistance?
response: Yes, We help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we ref...
Metadata: {'source': 'Do you provide any job assistance?', 'row': 11}
----------------------------------------
Result 2:
Content: prompt: Do you provide any virtual internship?
response: Yes...
Metadata: {'source': 'Do you provide any virtual internship?', 'row': 14}
----------------------------------------
Result 3:
Content: prompt: Will this bootcamp guarantee me a job?
response: The courses included in this bootcamp are done by 9000+ learners and many of them have secured a job which gives us ample confidence that you w...
Metadata: {'source': 'Will this bootcamp guarantee me a job?', 'row': 15}
----------------------------------------
Result 4:
Content: prompt: Will this course guarantee me a job?
response: We created a much lighter version of this course on YouT

As you can see above, the retriever that was created using FAISS and hugging face embedding is now capable of pulling relavant documents from our original CSV file knowledge store. This is very powerful and it will help us further in our project

##### Embeddings can be created using GooglePalm too. Also for vector database you can use chromadb as well as shown below. During our experimentation, we found hugging face embeddings and FAISS to be more appropriate for our use case

In [8]:
# google_palm_embeddings = GooglePalmEmbeddings(google_api_key=api_key)

# from langchain.vectorstores import Chroma
# vectordb = Chroma.from_documents(data,
#                            embedding=google_palm_embeddings,
#                            persist_directory='./chromadb')
# vectordb.persist()

### Create RetrievalQA chain along with prompt template 🚀

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from pprint import pprint

# 1. Define the prompt
prompt_template = """
                    Given the following context and a question, generate an answer
                    based on this context only.

                    In the answer, try to provide as much relevant information as possible
                    from the "response" section in the source document context without
                    making major changes.

                    If the answer is not found in the context, kindly state:
                    "I don't know."

                    Do not make up an answer.

                    CONTEXT:
                    {context}

                    QUESTION:
                    {input}
                """

# 1. Define the prompt
prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "input"]
)

# 2. Helper to format docs
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 3. LCEL with sources
chain = (
    RunnableParallel({
        "context": retriever | format_docs,
        "input": RunnablePassthrough(),
        "source_documents": retriever  # Keep raw docs for later
    })
    | RunnableParallel({
        "answer": prompt | llm | StrOutputParser(),
        "source_documents": lambda x: x["source_documents"]  # Pass through
    })
)

result = chain.invoke("Do you guys provide internship and also do you offer EMI payments?")
print(f"Answer: {result['answer']}")
pprint(f"Sources: {result['source_documents']}")

Answer: Yes, we provide virtual internships. However, we do not have an EMI option.
("Sources: [Document(id='3dd6b3dd-6ed9-499e-8dd8-6f8a2dd360e7', "
 "metadata={'source': 'Do we have an EMI option?', 'row': 13}, "
 "page_content='prompt: Do we have an EMI option?\\nresponse: No'), "
 "Document(id='203377be-d502-4dce-810f-9539a557eeb2', metadata={'source': 'Do "
 "you provide any virtual internship?', 'row': 14}, page_content='prompt: Do "
 "you provide any virtual internship?\\nresponse: Yes'), "
 "Document(id='8e1dc0ea-7d50-4df3-8c20-03f04b7c069a', metadata={'source': 'Do "
 "you provide any job assistance?', 'row': 11}, page_content='prompt: Do you "
 'provide any job assistance?\\nresponse: Yes, We help you with resume and '
 'interview preparation along with that we help you in building online '
 'credibility, and based on requirements we refer candidates to potential '
 "recruiters.'), Document(id='c8d4a883-1864-419a-988b-de1b9dc9a594', "
 "metadata={'source': 'Will this bootcamp

### We are all set 👍🏼 Let's ask some questions now

In [15]:
result = chain.invoke("Do you provide job assistance and also do you provide job gurantee?")
print(f"Answer: {result['answer']}")
pprint(f"Sources: {result['source_documents']}")

Answer: Yes, we help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we refer candidates to potential recruiters. However, we do not make any impractical promises regarding a guaranteed job. Our guarantee is to prepare you for the job market by teaching the most relevant skills, knowledge & timeless principles good enough to fetch the job.
("Sources: [Document(id='8e1dc0ea-7d50-4df3-8c20-03f04b7c069a', "
 "metadata={'source': 'Do you provide any job assistance?', 'row': 11}, "
 "page_content='prompt: Do you provide any job assistance?\\nresponse: Yes, We "
 'help you with resume and interview preparation along with that we help you '
 'in building online credibility, and based on requirements we refer '
 "candidates to potential recruiters.'), "
 "Document(id='203377be-d502-4dce-810f-9539a557eeb2', metadata={'source': 'Do "
 "you provide any virtual internship?', 'row': 14}, page_content='prompt: Do "
 "you

**As you can see above, the answer of question comes from two different FAQs within our csv file and it is able to pull those questions and merge them nicely**

In [17]:
chain("Do you guys provide internship and also do you offer EMI payments?")

{'query': 'Do you guys provide internship and also do you offer EMI payments?',
 'result': "Yes, we provide virtual internship and we don't offer EMI payments.",
 'source_documents': [Document(page_content='prompt: Do you provide any virtual internship?\nresponse: Yes', metadata={'source': 'Do you provide any virtual internship?', 'row': 14}),
  Document(page_content='prompt: Do we have an EMI option?\nresponse: No', metadata={'source': 'Do we have an EMI option?', 'row': 13}),
  Document(page_content='prompt: Do you provide any job assistance?\nresponse: Yes, We help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we refer candidates to potential recruiters.', metadata={'source': 'Do you provide any job assistance?', 'row': 11}),
  Document(page_content='prompt: How can I contact the instructors for any doubts/support?\nresponse: We have created every lecture with a motive to explain everything in an easy-

In [18]:
chain("do you have javascript course?")

{'query': 'do you have javascript course?',
 'result': "I don't know.",
 'source_documents': [Document(page_content='prompt: I have never done programming and belong to a non-technical background. Can I take this course?\nresponse: Yes, this is the perfect course for anyone who has never done coding and wants to build a career in the IT/Data Analytics industry or just wants to perform better in their current job or business using data.', metadata={'source': 'I have never done programming and belong to a non-technical background. Can I take this course?', 'row': 24}),
  Document(page_content='prompt: I have never done programming in my life. Can I take this bootcamp?\nresponse: Yes, this is the perfect bootcamp for anyone who has never done coding and wants to build a career in the IT/Data Analytics industry or just wants to perform better in your current job or business using data.', metadata={'source': 'I have never done programming in my life. Can I take this bootcamp?', 'row': 0}),


In [19]:
chain("Do you have plans to launch blockchain course in future?")

{'query': 'Do you have plans to launch blockchain course in future?',
 'result': "I don't know.",
 'source_documents': [Document(page_content='prompt: Will the course be upgraded when there are new features in Power BI?\nresponse: Yes, the course will be upgraded periodically based on the new features in Power BI, and learners who have already bought this course will have free access to the upgrades.', metadata={'source': 'Will the course be upgraded when there are new features in Power BI?', 'row': 27}),
  Document(page_content='prompt: What business concepts and domains are covered in this course?\nresponse: We have covered the core functions such as Sales, Marketing, Finance, and Supply Chain with their fundamentals related to this course. The domain you will learn in this course is consumer goods which is projected to have more openings and high data analytics requirements at least until 2030.', metadata={'source': 'What business concepts and domains are covered in this course?', '

In [20]:
chain("should I learn power bi or tableau?")

{'query': 'should I learn power bi or tableau?',
 'result': 'This is a contextual question. If you are talking about a pure visualization tool Tableau is slightly better. Data connectors, modeling and transformation features are available in both. However, factually speaking Power BI is cheaper and offers tighter integration with the Microsoft environment. Since most companies use excel & Microsoft tools they start with Power BI or move towards Power BI for seamless integration with other Microsoft tools (called as Power platform). This makes the job openings grow at a much higher rate on Power BI and Power Platform. Also, Power BI has been leading the Gartner’s magic quadrant in BI for the last few years as the industry leader.',
 'source_documents': [Document(page_content='prompt: Power BI or Tableau which one is better?\nresponse: This is a contextual question. If you are talking about a pure visualization tool Tableau is slightly better. Data connectors, modeling and transformation

In [21]:
chain("I've a MAC computer. Can I use powerbi on it?")

{'query': "I've a MAC computer. Can I use powerbi on it?",
 'result': 'response: Hi\n\nPower BI desktop works only in Windows OS. Please look into the system requirements section on this page. However, you can use a virtual machine to install and work with Power BI in other Operating systems.',
 'source_documents': [Document(page_content='prompt: How can I use PowerBI on my Mac system?\nresponse: Hi\n\nYou can use VirtualBox to create a virtual machine and install Windows on it. This will allow you to run Power BI and Excel on your Mac.\n\nIf you\'re not familiar with setting up a virtual machine, there are many resources available on YouTube that can guide you through the process. Simply search for "installing virtual machines" and you\'ll find plenty of helpful videos.\n\nBest of luck with your studies!', metadata={'source': 'How can I use PowerBI on my Mac system?', 'row': 44}),
  Document(page_content='prompt: Does Power BI work in Mac OS/Ubuntu?\nresponse: Power BI desktop works o

In [ ]:
chain("I don't see power pivot. how can I enable it?")

In [ ]:
chain("What is the price of your machine learning course?")